[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/solutions/50_mmdit_joint_attention_solution.ipynb)

# 🔴 Solution: MMDiT Joint Attention

Reference solution for `mmdit_joint_attention`.

In [ ]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math


In [ ]:
# ✅ SOLUTION

class MMDiTJointAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv_img = nn.Linear(dim, 3 * dim)
        self.qkv_txt = nn.Linear(dim, 3 * dim)
        self.proj_img = nn.Linear(dim, dim)
        self.proj_txt = nn.Linear(dim, dim)

    def _split_qkv(self, qkv, B, S):
        qkv = qkv.view(B, S, 3, self.num_heads, self.head_dim)
        return qkv.permute(2, 0, 3, 1, 4)

    def forward(self, image_tokens: torch.Tensor, text_tokens: torch.Tensor):
        B, S_img, D = image_tokens.shape
        S_txt = text_tokens.shape[1]
        q_i, k_i, v_i = self._split_qkv(self.qkv_img(image_tokens), B, S_img)
        q_t, k_t, v_t = self._split_qkv(self.qkv_txt(text_tokens), B, S_txt)
        q = torch.cat([q_i, q_t], dim=2)
        k = torch.cat([k_i, k_t], dim=2)
        v = torch.cat([v_i, v_t], dim=2)
        attn = torch.softmax((q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim), dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, S_img + S_txt, D)
        image_out, text_out = out[:, :S_img], out[:, S_img:]
        return self.proj_img(image_out), self.proj_txt(text_out)


In [ ]:
# Verify
attn = MMDiTJointAttention(dim=16, num_heads=4)
image = torch.randn(2, 6, 16)
text = torch.randn(2, 4, 16)
image_out, text_out = attn(image, text)
print(image_out.shape, text_out.shape)


In [ ]:
# Run judge
from torch_judge import check
check('mmdit_joint_attention')
